# Database Management Systems: Week 5 - In-Depth Notes

## Week 5 Overview: Relational Database Design Theory

Week 5 delves into the **formal theory of relational database design**, providing the mathematical foundation needed to create well-structured, anomaly-free database schemas. We begin by understanding the goals of good design and why naive tables lead to problems. Then, we introduce **functional dependencies** as the core concept for reasoning about data relationships. We develop **Armstrong's axioms** and the notion of closure to derive implied dependencies. We learn algorithms for computing **attribute closures**, detecting **extraneous attributes**, and finding **canonical covers**. Finally, we apply these tools to define **normal forms** (BCNF, 3NF) and to test decompositions for **lossless join** and **dependency preservation**.

This week is mathematically rigorous and essential for any serious database designer. The concepts learned here will be used directly in database implementation, query optimization, and maintenance.

---

## Module 21: Relational Database Design – Part 1: Goals, Redundancy, and 1NF

### 21.1. The Need for Good Relational Design

Given a set of attributes and real-world requirements, there are many possible ways to organize them into tables. Not all designs are equally good. We need a systematic approach to design "good" relational schemas.

**Key design goals:**
1. **Reflect real-world structure:** The schema must accurately model the entities, attributes, and relationships of the application domain.
2. **Adequacy for future data:** The design must be able to represent all anticipated data, including future additions and changes.
3. **Avoid redundant storage:** Minimize duplication of data to save space and prevent inconsistencies.
4. **Efficient access:** Queries should be answered quickly without excessive computation or storage overhead.
5. **Support data integrity:** The schema should allow the DBMS to enforce constraints (keys, foreign keys, domain constraints) effectively.
6. **Clean, consistent, and understandable:** The design should be easy for other developers and administrators to understand and maintain.

These goals are sometimes conflicting. For example, adding redundancy may improve query performance but harms storage and integrity. The designer must balance these trade-offs.

### 21.2. A Running Example: The Instructor-Department Table

Consider the original "instructor" relation and "department" relation from the university database:

```
instructor(ID, name, dept_name, salary)
department(dept_name, building, budget)
```

Suppose we combine them into a single table:

```
instructor_with_department(ID, name, dept_name, salary, building, budget)
```

This table is adequate (it holds all information) but has serious problems. Look at sample data: multiple instructors from the same department have the same `building` and `budget` values repeated. This is **redundancy**.

**Redundancy** occurs when the same piece of information is stored multiple times. It leads to:
- **Insertion Anomaly:** Cannot add a new instructor if the department's building or budget is unknown; you'd have to put NULL or invent values, which may violate constraints.
- **Deletion Anomaly:** If you delete the last instructor of a department, you lose the department's building and budget information, even though the department still exists.
- **Update Anomaly:** If a department's budget changes, you must update every row for that department. If you miss one, the database becomes inconsistent.

These anomalies are direct consequences of the **functional dependency** `dept_name → building, budget` while `dept_name` is not a key of the combined table. This dependency causes repetition.

### 21.3. Decomposition to Remove Redundancy

To eliminate redundancy and anomalies, we **decompose** the large table into smaller tables:

```
instructor(ID, name, dept_name, salary)
department(dept_name, building, budget)
```

This is **good decomposition** because:
- It minimizes dependencies that cause redundancy.
- It preserves all information (you can reconstruct the original via natural join).
- It honors functional dependencies.

**However, not every decomposition is good.** A bad decomposition can lead to **loss of information** or **inability to enforce dependencies**.

**Example of a lossy decomposition:**
Original: `employee(ID, name, street, city, salary)`
Decompose into:
- `employee1(ID, name)`
- `employee2(name, street, city, salary)`

If two employees have the same name, joining `employee1` and `employee2` on `name` can produce **extra spurious tuples** (cross-product of matching names) that were not in the original. This is called **lossy decomposition** because information (the exact associations) is lost. The resulting joined table has more tuples than the original, which is also a form of information loss.

### 21.4. Lossless Join Decomposition

A decomposition is **lossless join** if, for every possible instance of the original relation R, the natural join of the decomposed relations yields exactly the original R (no extra or missing tuples).

**Conditions for a binary decomposition** (R1, R2) to be lossless:
1. `R1 ∪ R2 = R` (the union covers all attributes).
2. `R1 ∩ R2 ≠ ∅` (there is at least one common attribute to join on).
3. `R1 ∩ R2` is a **superkey** of R1 **or** of R2 (or both). This ensures uniqueness on at least one side, preventing spurious combinations.

These are **sufficient** conditions (and in practice, they are also necessary for most cases). If condition 3 holds, the natural join reconstructs the original relation exactly.

### 21.5. Atomic Domains and First Normal Form (1NF)

**Atomic domain:** A domain where each value is considered indivisible. For example, an integer is atomic; a string that encodes multiple parts (like "CS101" where "CS" is the department) is **non-atomic** if you intend to split it. Similarly, a composite attribute like `name(first_name, middle_name, last_name)` is non-atomic.

**First Normal Form (1NF):** A relation is in 1NF if:
- The domain of every attribute is atomic.
- Every attribute is single-valued.

If a relation has composite or multi-valued attributes, it is **not in 1NF** and must be decomposed.

**Example:** A customer relation with a `phone_number` attribute that can hold multiple numbers (e.g., "123-456, 789-012") violates 1NF because it is multi-valued and composite.

**How to convert to 1NF:**
- **Option 1 (flattening):** Create separate attributes for each possible value (e.g., `phone1`, `phone2`). This is bad because it is arbitrary, leads to redundancy, and makes querying awkward.
- **Option 2 (separate table):** Create a new table `customer_phone(customer_id, phone_number)` with a foreign key to `customer`. This is the standard approach. The original relation becomes 1NF and the one-to-many relationship is properly modeled.

After converting to 1NF, we can proceed to higher normal forms (2NF, 3NF, BCNF) to remove further redundancies caused by functional dependencies.

---

## Module 22: Relational Database Design – Part 2: Functional Dependencies

### 22.1. Formal Definition of Functional Dependency (FD)

A **functional dependency** (FD) is a constraint between two sets of attributes in a relation. Given a relation schema R and two subsets of attributes α (alpha) and β (beta), we write:

```
α → β
```

(read: "α functionally determines β" or "β is functionally dependent on α").

**Formal condition:** FD `α → β` holds on R if and only if, for any legal instance r of R, and for any two tuples t1 and t2 in r, if t1[α] = t2[α], then t1[β] = t2[β].

In simple terms: if two tuples agree on the values of α, they must also agree on the values of β.

**Key points:**
- FDs are **constraints derived from the business rules**, not from a particular instance of data.
- FDs must hold for **all possible instances** (past, present, future).
- FD is a generalization of the key concept: a superkey K functionally determines all attributes of R (K → R).

**Example:** In `instructor_with_department`, we have `dept_name → building` and `dept_name → budget`. This means that given a department name, the building and budget are uniquely determined. Two instructors from the same department must have the same building and budget.

**Non-example:** `dept_name → salary` does **not** hold because two instructors in the same department can have different salaries.

### 22.2. Trivial Functional Dependencies

A functional dependency `α → β` is **trivial** if β ⊆ α. Since if two tuples agree on all attributes of α, they certainly agree on any subset of α, the dependency is always satisfied.

**Examples:**
- `{ID, name} → {ID}` is trivial.
- `{dept_name, building} → {dept_name}` is trivial.

### 22.3. Armstrong's Axioms

Armstrong's axioms are a sound and complete set of inference rules for functional dependencies. They allow us to derive all FDs logically implied by a given set F.

**Axioms:**
1. **Reflexivity:** If β ⊆ α, then α → β.
2. **Augmentation:** If α → β, then αγ → βγ for any set of attributes γ.
3. **Transitivity:** If α → β and β → γ, then α → γ.

**Soundness:** All FDs derived using these axioms are logically valid (they hold in any relation satisfying the original FDs).

**Completeness:** Any FD that is logically implied by a given set F can be derived using only these three axioms. In other words, repeated application of the axioms generates exactly the closure F+.

### 22.4. Closure of a Set of FDs (F+)

The **closure** of a set F of FDs, denoted F+, is the set of all FDs that are logically implied by F. It includes F itself and all trivial FDs and all FDs derived via Armstrong's axioms.

**Example:** If F = {A → B, B → C}, then F+ includes:
- A → B, B → C (given)
- A → C (by transitivity)
- A → A, B → B, C → C, AB → A, AB → B, etc. (reflexivity)
- Many more trivial ones.

F+ can be infinite, but we usually work with a finite **canonical cover**.

### 22.5. Computing F+ (Naive Algorithm)

A naive algorithm to compute F+:
1. Start with F+ = F.
2. Repeat until no change:
   - For each FD in F+, apply reflexivity and augmentation to generate new FDs, add them to F+.
   - For each pair of FDs in F+ where the right-hand side of one equals the left-hand side of the other, apply transitivity, add result.
3. Terminate when F+ does not change.

This is guaranteed to terminate because the number of possible FDs is finite (though exponential in the number of attributes).

### 22.6. Derived Rules

From Armstrong's axioms, we can derive additional rules that are convenient:

- **Union:** If α → β and α → γ, then α → βγ.
- **Decomposition:** If α → βγ, then α → β and α → γ.
- **Pseudo-transitivity:** If α → β and γβ → δ, then γα → δ.

These rules are sound but not necessary for completeness (the three basic axioms are sufficient).

---

## Module 23: Relational Database Design – Part 3: Attribute Closure, BCNF, and 3NF

### 23.1. Attribute Closure (α+)

The **closure of a set of attributes α under a set of FDs F**, denoted α+, is the set of all attributes that are functionally determined by α. That is, α+ = {A | α → A is in F+}.

**Algorithm:**
```
result = α
repeat until no change:
    for each FD β → γ in F:
        if β ⊆ result:
            result = result ∪ γ
return result
```

This algorithm is efficient (polynomial time) and is fundamental for many design tasks.

**Example from transcript:**
Given R(A,B,C,G,H,I) and F = {A→B, A→C, CG→H, CG→I, B→H}.
Compute (AG)+:
- Start: result = {A,G}
- A→B: A ∈ result → add B. result = {A,G,B}
- A→C: A ∈ result → add C. result = {A,G,B,C}
- CG→H: CG ⊆ result? C and G are in result → add H. result = {A,G,B,C,H}
- CG→I: CG ⊆ result → add I. result = {A,G,B,C,H,I}
- B→H: B ∈ result, H already present.
Final (AG)+ = {A,G,B,C,H,I} = entire R. So AG is a superkey.

### 23.2. Uses of Attribute Closure

- **Testing superkey:** α is a superkey if α+ = R (all attributes).
- **Testing candidate key:** α is a candidate key if α+ = R and for no proper subset β ⊂ α is β+ = R.
- **Testing functional dependency:** α → β holds iff β ⊆ α+.
- **Computing F+:** For each subset γ of R, compute γ+; then for each S ⊆ γ+, add FD γ → S to F+.

### 23.3. Extraneous Attributes

An attribute in a functional dependency may be **extraneous** (redundant) if it can be removed without changing the closure of the FD set.

**Definition:** Given F containing α → β, an attribute A is extraneous if:
- **Left side (A ∈ α):** If (α - A) → β is implied by F (i.e., (α - A)+ under F contains β), then A is extraneous in α.
- **Right side (A ∈ β):** If α → (β - A) is implied by F (i.e., α+ under F' (where F' replaces α→β with α→(β-A)) contains A), then A is extraneous in β.

**Tests using attribute closure:**
- For left: compute (α - A)+ under F; if it includes β, A is extraneous.
- For right: compute α+ under F' (replace α→β with α→(β-A)); if it includes A, A is extraneous.

### 23.4. Canonical Cover (Minimal Cover)

A **canonical cover** Fc of a set F of FDs is a minimal set of FDs such that Fc+ = F+. It is unique up to attribute order and union of left-hand sides.

**Properties:**
1. Fc is equivalent to F (Fc+ = F+).
2. No FD in Fc contains an extraneous attribute on either side.
3. Each left-hand side of an FD in Fc is unique (no two FDs have the same left side).

**Algorithm:**
1. Use union rule to combine FDs with same left side.
2. For each FD, remove extraneous attributes from left and right using attribute closure tests.
3. Repeat until no change.
4. Eliminate redundant FDs (those that can be derived from others).

### 23.5. Boyce-Codd Normal Form (BCNF)

A relation R is in **BCNF** with respect to a set F of FDs if for every FD α → β in F+ (the closure), at least one of the following holds:
- α → β is trivial (β ⊆ α), **or**
- α is a **superkey** of R.

In other words, the only non-trivial FDs allowed are those where the left side is a superkey. This ensures that every determinant is a key, eliminating redundancy.

**Example:** `instructor_with_department` violates BCNF because `dept_name → building` is non-trivial and `dept_name` is not a superkey.

**Decomposition to BCNF:**
If a relation violates BCNF due to α → β, decompose into:
- R1 = α ∪ β
- R2 = R - (β - α)

Repeat until all relations are in BCNF.

### 23.6. Third Normal Form (3NF)

A relation R is in **3NF** if for every FD α → β in F+, at least one of the following holds:
- α → β is trivial, **or**
- α is a superkey, **or**
- Every attribute in β - α is contained in some **candidate key** of R.

The third condition relaxes BCNF to allow some redundancy as long as it is tied to a candidate key. This ensures **dependency preservation** (which BCNF may not guarantee).

**Example:** CSZ(city, street, zip) with FDs {CS → Z, Z → C}. CS is a candidate key. Z → C is non-trivial and Z is not a superkey, so BCNF violation. But Z → C has C in β-α = {C}, and C is part of the candidate key CS? Actually C is not in CS; CS = {city, street}. So C is not part of CS. This relation is **not in 3NF**? Actually the usual example shows 3NF allows Z→C because C is a prime attribute (part of some candidate key). But here candidate key is CS; C alone is not a candidate key or part of a candidate key. That seems off. Wait, the transcript's example: CS determines Z, Z determines C. The candidate key is CS? If CS is the only candidate key, then C is non-prime. So Z→C violates 3NF? But the transcript says it is in 3NF? Let's re-read: transcript says "CS determines that, city and street determines as a put, and zip code determines the city, these are the functional dependencies obviously, CS is the key." Then it says "Now, CS determines Z is in BCNF, but Z determines C violates because it is neither trivial nor Z is a key." So they decompose. Then they show the decomposition is lossless but not dependency preserving? Actually they say BCNF decomposition may not be dependency preserving, and then they introduce 3NF as a relaxation that is dependency preserving. They don't explicitly state this example is 3NF; they use it to show BCNF issue. In 3NF, we allow Z → C because C is a prime attribute? Actually the standard example: CSZ with CS→Z and Z→C, candidate key CS, so C is non-prime. So Z→C would violate 3NF because C is not prime. So it's actually not in 3NF. The professor's point was to show BCNF decomposition may not preserve dependencies; then 3NF is introduced later. I think the transcript is a bit loose. I'll clarify the standard definition.

**Definition of 3NF:** For every non-trivial FD α → β, either α is a superkey, or every attribute in β-α is a **prime attribute** (member of some candidate key). This allows some redundancy but preserves dependencies.

### 23.7. Dependency Preservation

A decomposition is **dependency preserving** if the union of the projections of F+ onto the decomposed schemas is equivalent to F+. That is, all original FDs can be checked on the individual decomposed relations without needing to join.

**Importance:** Checking dependencies after decomposition is efficient if each FD can be verified locally. If a join is required, performance suffers.

**Example:** Decomposing CSZ into ZC and SZ; the FD CS→Z cannot be checked locally, so this decomposition is **not dependency preserving**. 3NF is designed to be both lossless join and dependency preserving (whereas BCNF may sacrifice dependency preservation).

### 23.8. Motivation for Higher Normal Forms (4NF)

BCNF and 3NF handle redundancy caused by functional dependencies. But redundancy can also arise from **multi-valued dependencies** (MVDs). Consider `instructor_info(ID, child_name, phone)` with no non-trivial FDs; it is in BCNF but still has redundancy because one instructor may have multiple children and multiple phones, causing cross-product duplication. Such relations need **Fourth Normal Form (4NF)**, which we will study later.

---

## Module 24: Relational Database Design – Part 4: Algorithms for FD Theory

### 24.1. Recap: Attribute Closure Algorithm

We've already seen the algorithm; here we emphasize its computational role.

**Algorithm:**
```
def closure(alpha, F):
    result = alpha
    changed = True
    while changed:
        changed = False
        for (beta, gamma) in F:
            if beta.issubset(result) and not gamma.issubset(result):
                result = result.union(gamma)
                changed = True
    return result
```

**Complexity:** Polynomial in the size of F and number of attributes.

**Uses:**
- Superkey test: alpha+ == R
- Candidate key: superkey and no proper subset is superkey
- FD membership: alpha → beta holds iff beta ⊆ alpha+
- Compute F+ (by iterating over all subsets).

### 24.2. Detecting Extraneous Attributes

**Left-side extraneous attribute:** To test if A ∈ α is extraneous in α → β:
1. Let α' = α - {A}.
2. Compute α'+ under F (original F).
3. If β ⊆ α'+, then A is extraneous.

**Right-side extraneous attribute:** To test if A ∈ β is extraneous in α → β:
1. Let β' = β - {A}.
2. Let F' be F with α→β replaced by α→β'.
3. Compute α+ under F'.
4. If A ∈ α+, then A is extraneous.

**Examples from transcript:**
- F = {A→C, AB→C}. Check B extraneous in AB→C. Compute A+ = {A,C}. Since C ∈ A+, B is extraneous (AB→C can be reduced to A→C, which is already in F).
- F = {A→C, AB→CD}. Check C extraneous in AB→CD? Compute AB+ under F' (replace with AB→D). F' = {A→C, AB→D}. AB+ = {A,B,C,D}. Since C ∈ AB+, C is extraneous.

### 24.3. Equivalence of FD Sets

Two sets F and G are **equivalent** if F+ = G+.

To test equivalence:
- **F covers G:** For each FD α→β in G, compute α+ under F and check β ⊆ α+.
- **G covers F:** For each FD α→β in F, compute α+ under G and check β ⊆ α+.
If both covers hold, they are equivalent.

### 24.4. Canonical Cover Algorithm

```
Fc = F
repeat:
    Use union rule to combine FDs with same left side.
    Find an FD α→β in Fc with an extraneous attribute.
    If found, remove that attribute.
until no change
```

**Note:** The order of operations may yield different canonical covers, but all are minimal and equivalent.

**Example:**
F = {A→B, B→C, A→C}. 
- A→C is redundant (implied by A→B and B→C). Remove it.
- Fc = {A→B, B→C}.

F = {A→B, B→C, A→CD}.
- C in A→CD is extraneous because A→B and B→C imply A→C.
- Fc = {A→B, B→C, A→D}.

### 24.5. Practice Problems

- **Superkeys:** Given F, find all superkeys.
- **Candidate keys:** Among superkeys, find minimal ones.
- **Prime vs non-prime attributes:** Prime attributes are those that are part of some candidate key.
- **Equivalence of FD sets.**
- **Minimal covers.**

These exercises solidify the understanding of FD theory.

---

## Module 25: Relational Database Design – Part 5: Lossless Join and Dependency Preservation

### 25.1. Lossless Join Decomposition: Detailed

We formalize the conditions for a binary decomposition to be lossless.

**Given:** R decomposed into R1 and R2.

**Sufficient conditions for lossless join:**
1. R1 ∪ R2 = R.
2. R1 ∩ R2 ≠ ∅.
3. R1 ∩ R2 → R1 (or R2) is in F+ (i.e., the common attributes are a superkey of R1 or R2).

If these hold, then for any instance r of R, r = π_R1(r) ⋈ π_R2(r).

**Intuition:** If the common attributes are a key on one side, then each tuple in that side is uniquely identified. Joining with the other side cannot create spurious combinations because there is only one match per key value.

**Example from transcript:** Supplier(Supplier_ID, Name, City, Part, Quantity) decomposed into (Supplier_ID, Name, City) and (Supplier_ID, Part, Quantity). The intersection is Supplier_ID, which is a key in the first relation. So lossless.

A bad decomposition where intersection is Quantity (not a key) produces spurious tuples.

### 25.2. Dependency Preservation: Detailed

**Definition:** A decomposition R1,...,Rn is dependency preserving if the union of the projections of F+ onto each Ri is equivalent to F.

**Formally:** Let Fi be the set of FDs in F+ that involve only attributes of Ri. Then the decomposition is dependency preserving if (∪Fi)+ = F+.

**Why important:** We want to enforce FDs locally. If a dependency spans multiple relations, enforcing it requires a join, which is expensive and may violate integrity during concurrent updates.

**Testing dependency preservation (naive):**
1. Compute F+.
2. For each Ri, compute Fi = {α→β in F+ | αβ ⊆ Ri}.
3. Compute closure of ∪Fi.
4. If it equals F+, then decomposition is dependency preserving.

This is exponential because F+ can be huge. A better algorithm uses **attribute closure**.

**Polynomial-time algorithm to test if α→β is preserved:**
```
result = α
repeat:
    for each Ri:
        t = closure(result ∩ Ri) under F
        t = t ∩ Ri
        result = result ∪ t
until no change
if β ⊆ result: α→β is preserved
else: not preserved
```

**Intuition:** We build up the closure of α using only attributes that can be checked within each decomposed relation. If we can eventually include β, then the dependency can be enforced by checking local FDs.

### 25.3. Worked Examples

**Example 1:** R(A,B,C,D) with F = {A→B, B→C, C→D, D→A}. Decompose into R1(A,B), R2(B,C), R3(C,D). Is the decomposition dependency preserving? 
- F1 = {A→B, B→A} (from closure), F2 = {B→C, C→B}, F3 = {C→D, D→C}.
- Union gives all needed dependencies; check D→A: using F1,F2,F3, we can infer D→C→B→A. So yes, dependency preserving.

**Example 2:** R(A,B,C) with F = {A→B, B→C}. Decompose into R1(A,B) and R2(A,C). This is lossless because A is a key in both. But B→C cannot be checked locally because neither R1 nor R2 contains both B and C. So not dependency preserving.

### 25.4. Summary

We have now a robust toolkit for relational database design:
- Understand good design goals.
- Use functional dependencies to capture business rules.
- Apply Armstrong's axioms to reason about dependencies.
- Compute attribute closures and canonical covers to minimize dependencies.
- Test normal forms (BCNF, 3NF) to eliminate anomalies.
- Ensure decompositions are lossless join and dependency preserving.

In subsequent weeks, we will extend this theory to multi-valued dependencies and higher normal forms (4NF, 5NF), and then move to physical database design.

---

This concludes the in-depth notes for Week 5. Each module has been expanded with formal definitions, intuitive explanations, algorithms, and examples to provide a comprehensive understanding of relational database design theory.